<a href="https://colab.research.google.com/github/klay-10/RFM-customer-segmentation-/blob/main/RFM_Analysis_using_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#import libraries

In [313]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
import plotly.express as px



#load dataset

In [314]:
df = pd.read_csv('/content/drive/MyDrive/bakery/bakery_sales_dataset.csv')


 #Exploratory Data Analysis

### Do not clean the data immediately. First, inspect it. This helps you understand what must be fixed.

###Check shape, first rows

In [315]:
print("First 5 rows of the dataset:")
print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

First 5 rows of the dataset:
| Order_ID   | Customer_ID   | Customer_Age   | Customer_Gender   | Customer_Segment   | Loyalty_Member   | Date       | Year   | Month   | Day   | Day_of_Week   | Weekend   | Hour   | Time   | Season   | Festival   | Weather   | Temperature   | Product        | Category       | Quantity   | Unit_Price   | Cost_Price   | Discount_Percentage   | Selling_Price   | Total_Bill   | Payment_Method   | Promotion_Applied   | Promotion_Type   | Shelf_Life_Days   | Storage_Type     | Perishability   | Store_ID   | Units_Produced   | Units_Sold   | Unsold_Units   | Stock_Available   | Waste_Pct   | Profit   | Customer_Rating   |
|:-----------|:--------------|:---------------|:------------------|:-------------------|:-----------------|:-----------|:-------|:--------|:------|:--------------|:----------|:-------|:-------|:---------|:-----------|:----------|:--------------|:---------------|:---------------|:-----------|:-------------|:-------------|:----------------------

###check data types and memory structure

In [316]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19321 entries, 0 to 19320
Data columns (total 40 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Order_ID             19321 non-null  object 
 1   Customer_ID          19321 non-null  object 
 2   Customer_Age         19321 non-null  int64  
 3   Customer_Gender      19321 non-null  object 
 4   Customer_Segment     19321 non-null  object 
 5   Loyalty_Member       19321 non-null  object 
 6   Date                 19321 non-null  object 
 7   Year                 19321 non-null  int64  
 8   Month                19321 non-null  int64  
 9   Day                  19321 non-null  int64  
 10  Day_of_Week          19321 non-null  object 
 11  Weekend              19321 non-null  object 
 12  Hour                 19321 non-null  int64  
 13  Time                 19321 non-null  object 
 14  Season               19321 non-null  object 
 15  Festival             3315 non-null  

###Check duplicate records

In [317]:
print('Number of duplicate rows:', df.duplicated().sum())

Number of duplicate rows: 0


###Summary statistics

In [318]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Customer_Age,19321.0,35.541846,13.733832,16.00,25.00,34.00,45.00,73.00
Year,19321.0,2023.508100,0.499947,2023.00,2023.00,2024.00,2024.00,2024.00
Month,19321.0,6.559184,3.478048,1.00,4.00,7.00,10.00,12.00
Day,19321.0,15.871901,8.767930,1.00,8.00,16.00,24.00,31.00
Hour,19321.0,12.864810,3.940440,7.00,9.00,12.00,17.00,21.00
Temperature,19321.0,26.988127,8.813511,8.10,21.20,27.70,33.20,44.90
Quantity,19321.0,2.226955,1.782968,1.00,1.00,2.00,3.00,20.00
Unit_Price,19321.0,87.263624,110.481951,7.60,31.02,45.09,76.65,472.42
Cost_Price,19321.0,49.275521,65.969024,3.27,16.90,23.64,52.64,296.28
Discount_Percentage,19321.0,4.345541,6.794384,0.00,0.00,0.00,7.50,30.00


In [319]:
df.isnull().sum()

,0
Order_ID,0
Customer_ID,0
Customer_Age,0
Customer_Gender,0
Customer_Segment,0
Loyalty_Member,0
Date,0
Year,0
Month,0
Day,0


the needed features for this project have zero values, so we good to go.

###Inspect unique features

In [320]:
df.nunique()

,0
Order_ID,6576
Customer_ID,500
Customer_Age,57
Customer_Gender,3
Customer_Segment,4
Loyalty_Member,2
Date,731
Year,2
Month,12
Day,31


### To ensure we have a purchasedate column which is essential for calculating recency we combine date and time

In [321]:
df['Purchase_Date'] = pd.to_datetime(
    df['Date'].astype(str) + ' ' + df['Time'].astype(str)
)


#Calculate Recency, Frequency, and Monetary Values

In [322]:
# get the latest date in the dataset and add one day to serve as the snapshot date
snapshot_date = df['Purchase_Date'].max() + pd.Timedelta(days=1)

# calculate RFM values for each customer
rfm_df = df.groupby('Customer_ID').agg(
    Recency=('Purchase_Date', lambda date: (snapshot_date - date.max()).days),
    Frequency=('Order_ID', 'count'),
    Monetary=('Total_Bill', 'sum')
).reset_index()

print("Calculated RFM values:")
print(rfm_df.head().to_markdown(index=False, numalign="left", stralign="left"))

Calculated RFM values:
| Customer_ID   | Recency   | Frequency   | Monetary   |
|:--------------|:----------|:------------|:-----------|
| CUST0001      | 132       | 19          | 3315.5     |
| CUST0002      | 130       | 34          | 5205.51    |
| CUST0003      | 52        | 26          | 2835.69    |
| CUST0004      | 175       | 56          | 7682.36    |
| CUST0005      | 4         | 50          | 5153.83    |


# we plot and visualize how recently customers have made transactions

In [323]:
fig = go.Figure()

# Calculate median recency
median_recency = rfm_df['Recency'].median()

# Customer count histogram
fig.add_trace(go.Histogram(
    x=rfm_df['Recency'],
    nbinsx=30,
    name='Customer Count',
    marker_color='skyblue',
    opacity=0.75
))

# Median reference line
fig.add_vline(
    x=median_recency,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Median: {median_recency:.1f} days',
    annotation_position='top right'
)

fig.update_layout(
    title=dict(
        text='Customer Recency Distribution',
        x=0.5,
        xanchor='center',
        font=dict(size=20)
    ),
    xaxis_title='Recency (Days since last purchase)',
    yaxis_title='Number of Customers',
    template='plotly_white',
    bargap=0.1,
    showlegend=True
)

fig.show()


###Observations
From the graph it's right skewed which means many customers purchased recently with peak purchase around the first 20 days, as the number of days increases, the number of customers generally decreases.Fewer customers have gone longer without purchase. Hence has to investigate the inactive customers for more answers.
###BUSINESS IMPLICATIONS:
* Customers with low recency values should receive loyalty rewards and personalized offers.
* Customers approaching 44 days without purchasing should receive targeted engagement campaigns.
* The business should introduce automated reminders and win-back campaigns.

NB:
I use median as determine factor because it is not affected by outliners.

#check how often customers buy

In [324]:
freq = rfm_df['Frequency']

pg80,med = np.percentile(freq,80),np.median(freq)
fig = px.histogram(
    rfm_df,
    x='Frequency',
    nbins=30,
    marginal='box',
    title='How Often Do Customers Buy? (Spot the Loyalists)',
    labels={'Frequency': 'Purchases per Customer'},
    template='plotly_white',
    opacity=0.9
)
fig.update_traces(
    hovertemplate='Purchases: %{x}<br>Customers: %{y}'
)
fig.add_vline(
    x=med,
    line_dash='dash',
    line_width=2,
    annotation_text=f"Median = {med:.0f}",
    annotation_position='top left'
)
fig.add_vrect(
    x0=pg80,
    x1=freq.max(),
    opacity=0.1,
    line_width=0,
    annotation_text='Top 20% Loyalists',
    annotation_position='top right'
)
fig.add_annotation(
    x=freq.min(),
    y=0,
    yshift=40,
    showarrow=False,
    text='Left tail: one-time/rare buyers → retention opportunities'
)
fig.update_layout(
    yaxis_title='Number of Customers'
)
fig.show()

###Observations
* The median customer makes approximately 38 purchases. This suggests relatively frequent purchasing activity within the observed dataset.
* A large proportion of customers fall within 30–45 purchases, indicating that purchasing frequency is concentrated around the median.
* The left tail represents customers who have made relatively few purchases. These could be occasional buyers or customers who have not yet developed purchasing habits.
* The right tail contains customers with substantially more purchases than the typical customer.
* A few customers make approximately 75–80 purchases. These customers exhibit unusually high purchasing frequency and may represent valuable loyalty segments.
###Business implications
* Reward high-frequency customers through loyalty programmes.
* Investigate what motivates frequent customers to keep purchasing.
* Encourage low-frequency customers through personalized promotions.
* Develop retention strategies to move occasional buyers towards repeat purchasing.

#we check the amount spent per customer

In [325]:
rfm_df['Monetary'].describe().T

,Monetary
count,500.000000
mean,5381.612100
std,2196.259224
min,779.800000
25%,3816.167500
50%,5109.895000
75%,6870.842500
max,14727.430000


In [326]:
monetary = rfm_df['Monetary'].dropna()

q80 = np.percentile(monetary, 80)
median = np.median(monetary)

fig = px.histogram(
    rfm_df,
    x='Monetary',
    nbins=30,
    marginal='box',
    title='How Much Do Customers Spend? (Identify High-Value Customers)',
    labels={'Monetary': 'Total Spending per Customer'},
    template='plotly_white',
    opacity=0.9
)

fig.update_traces(
    hovertemplate='Spending: %{x}<br>Customers: %{y}'
)

fig.add_vline(
    x=median,
    line_dash='dash',
    line_width=2,
    line_color='red',
    annotation_text=f'Median = {median:,.2f}',
    annotation_position='top left'
)

fig.add_vrect(
    x0=q80,
    x1=monetary.max(),
    opacity=0.1,
    line_width=0,
    annotation_text='Top 20% spending threshold',
    annotation_position='top right'
)

fig.update_layout(
    yaxis_title='Number of Customers',
    xaxis_title='Total Monetary Value',
    title_x=0.5
)

fig.show()


###observations
* It is right-skewed. Most customers cluster between roughly 2k and 8k, with a long tail stretching out to about 14k+.
* The peak (mode) is around 4k, where the tallest bar reaches about 50 customers.
* The median is 5,109.90, marked by the red dashed line. Half of customers spend less than this and half spend more.
* The median sits to the right of the peak, which is typical of right-skewed data. The mean is probably higher than the median, pulled up by big spenders.
* The whisker extends to about 12k. The few dots beyond it (around 12.5k to 15k) are outliers, meaning a handful of high spenders.
* The few customers in the 12k-15k range are  VIP tier. There are very few of them, but they likely contribute disproportionately to revenue.
###Business Implications
1. Revenue is not evenly spread. A small group of high spenders probably drives a large share of total value, so retention efforts should prioritize them.
2. The mid-range customers (4k-6k) are the bulk of the base. Loyalty campaigns could push them toward the top-20% threshold.
3. The low-spend left tail (under 2k) is small, so there isn’t a big pool of very low-value customers.


#segment our customers

In [327]:
# define scoring functions
def r_score(value, r_quartiles):
    if value <= r_quartiles[0.25]:
        return 4
    elif value <= r_quartiles[0.5]:
        return 3
    elif value <= r_quartiles[0.75]:
        return 2
    else:
        return 1

def f_score(value, f_quartiles):
    if value <= f_quartiles[0.25]:
        return 1
    elif value <= f_quartiles[0.5]:
        return 2
    elif value <= f_quartiles[0.75]:
        return 3
    else:
        return 4

def m_score(value, m_quartiles):
    if value <= m_quartiles[0.25]:
        return 1
    elif value <= m_quartiles[0.5]:
        return 2
    elif value <= m_quartiles[0.75]:
        return 3
    else:
        return 4

# get quartiles
r_quartiles = rfm_df['Recency'].quantile([0.25, 0.5, 0.75])
f_quartiles = rfm_df['Frequency'].quantile([0.25, 0.5, 0.75])
m_quartiles = rfm_df['Monetary'].quantile([0.25, 0.5, 0.75])

# apply scoring functions to create RFM scores
rfm_df['R_Score'] = rfm_df['Recency'].apply(lambda x: r_score(x, r_quartiles))
rfm_df['F_Score'] = rfm_df['Frequency'].apply(lambda x: f_score(x, f_quartiles))
rfm_df['M_Score'] = rfm_df['Monetary'].apply(lambda x: m_score(x, m_quartiles))

rfm_df['RFM_Score'] = rfm_df['R_Score'].astype(str) + rfm_df['F_Score'].astype(str) + rfm_df['M_Score'].astype(str)

#After scoring, we combine them into a single RFM_Score string, like ‘444’ or ‘111’:

In [328]:
rfm_df[['Customer_ID', 'Recency', 'Frequency', 'Monetary',
        'R_Score', 'F_Score', 'M_Score', 'RFM_Score']].head(10)

,Customer_ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,CUST0001,132,19,3315.50,1,1,1,111
1,CUST0002,130,34,5205.51,1,2,3,123
2,CUST0003,52,26,2835.69,2,1,1,211
3,CUST0004,175,56,7682.36,1,4,4,144
4,CUST0005,4,50,5153.83,4,4,3,443
5,CUST0006,62,47,5661.05,2,4,3,243
6,CUST0007,12,36,5128.97,4,2,3,423
7,CUST0008,78,31,5192.23,2,2,3,223
8,CUST0009,57,32,4415.80,2,2,2,222
9,CUST0010,52,47,7158.30,2,4,4,244


###first we check the unique values

In [329]:
rfm_df['RFM_Score'].unique()

array(['111', '123', '211', '144', '443', '243', '423', '223', '222',
       '244', '344', '311', '233', '444', '112', '433', '312', '234',
       '422', '333', '143', '122', '323', '133', '412', '242', '432',
       '424', '322', '321', '332', '212', '121', '232', '434', '224',
       '411', '221', '213', '132', '334', '343', '421', '342', '113',
       '134', '442', '413', '313'], dtype=object)

#applying rule base segment

In [330]:

def rfm_segment(row):

    R = row['R_Score']
    F = row['F_Score']
    M = row['M_Score']

    if R >= 3 and F >= 3 and M >= 3:
        return 'High Value Customers'

    elif R >= 2 and F >= 3 and M >= 2:
        return 'Loyal Customers'

    elif R >= 3 and F <= 2:
        return 'Potential New Customers'

    elif R <= 2 and F >= 3 and M >= 3:
        return 'At Risk'

    elif R == 1 and F <= 2 and M <= 2:
        return 'Lost Customers'

    else:
        return 'Other'


rfm_df['Segment'] = rfm_df.apply(rfm_segment, axis=1)
rfm_df['Segment'].value_counts()

,count
Segment,
High Value Customers,120
Potential New Customers,108
Loyal Customers,90
Lost Customers,75
Other,75
At Risk,32


####This will show you which RFM combinations you have not explicitly classified.

In [331]:
rfm_df[rfm_df['Segment'] == 'Other']['RFM_Score'].value_counts()

,count
RFM_Score,
211,26
222,11
223,10
132,8
123,6
221,5
212,4
224,2
213,2


###visually compare segments

In [332]:

import plotly.express as px

segment_counts = (
    rfm_df['Segment']
    .value_counts()
    .reset_index()
)

segment_counts.columns = ['Segment', 'Customers']

fig = px.bar(
    segment_counts,
    x='Segment',
    y='Customers',
    title='Customer Distribution by RFM Segment',
    text='Customers',
    template='plotly_white'
)

fig.update_layout(
    title_x=0.5,
    xaxis_title='Customer Segment',
    yaxis_title='Number of Customers'
)

fig.show()


#customer segment summary

In [333]:
seg = (
    rfm_df['Segment'].value_counts()
    .rename_axis('Segment')
    .reset_index(name='Customers')
)

seg['Share'] = (
    seg['Customers'] / seg['Customers'].sum() * 100
).round(1)
top = seg.iloc[0]['Segment']

seg['Highlight'] = np.where(
    seg['Segment'].eq(top),
    'Top segment',
    'Others'
)

fig = px.bar(
    seg,
    x='Segment',
    y='Customers',
    color='Highlight',
    custom_data=['Share'],
    text=seg.apply(
        lambda r: f"{int(r.Customers)} ({r.Share}%)",
        axis=1
    ),
    title='Customer Segments: Where Is The Value?',
    labels={'Customers':'Number of Customers'},
    template='plotly_white',
    opacity=0.95
)
fig.update_traces(
    textposition='outside',
    hovertemplate=(
    '<b>%{x}</b><br>'
    'Customers: %{y}<br>'
    'Share: %{customdata[0]}%'
    '<extra></extra>'
)
)
fig.update_layout(
    xaxis_title='RFM Segment',
    xaxis={
        'categoryorder':'array',
        'categoryarray':seg['Segment']
    },
    showlegend=False,
    margin=dict(t=70, b=20)
)
top_y = int(
    seg.loc[seg['Segment'].eq(top), 'Customers'].iloc[0]
)

recommendations = {
    'Champions': 'Protect high-value customers through retention and rewards',
    'Loyal Customers': 'Strengthen loyalty and encourage higher spending',
    'New Customers': 'Encourage second purchases through onboarding',
    'At Risk': 'Launch win-back campaigns to prevent churn',
    'Lost Customers': 'Test targeted reactivation campaigns',
    'Other': 'Investigate unclassified customer behaviour'
}

fig.add_annotation(
    x=top,
    y=top_y,
    yshift=35,
    showarrow=True,
    text=recommendations.get(
        top,
        'Investigate this customer segment'
    )
)
fig.show()

 High value customers are the largest segment, the business should prioritise retaining them and increasing their lifetime value.

#Final words
RFM Analysis is a simple tool i recommend every business owner should have in their arsenal. It moves you from generic, all in one marketing to innovative, data-driven strategies that genuinely connect with your customers.

#Questions to reflect beyond the Data
Data is valuable, but its real power lies in the decisions, solutions, and opportunities it creates. My goal is to continue exploring how data science, machine learning, and artificial intelligence can solve practical problems and create meaningful impact.

- Business: Are your most valuable customers really the ones who spend the most?
- Technology: Should we trust AI decisions we cannot explain?
- Ethics: Can an accurate model still be unfair?
- Impact: What makes a data science project truly successful?